# Task 3 - LinearSVC

Task 3 implementation using TF-IDF features with `LinearSVC` for multi-class classification of ArXiv CS paper abstracts.

The notebook uses a local train/validation split for model selection, then retrains the best configuration on the full training set and saves a Kaggle-format prediction CSV.

In [1]:
# Section 0 - Imports & Config
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

CONFIG = {
    'TRAIN_PATH': 'dataset/train.csv',
    'TEST_PATH': 'dataset/test.csv',
    'SUBMISSIONS_DIR': 'submissions',
    'RANDOM_STATE': 42,
    'VAL_SIZE': 0.2,
    'BASELINE_TFIDF': {
        'max_features': 5000,
        'ngram_range': (1, 1),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': True,
    },
    'TUNING_GRID': [
        {
            'name': 'baseline_unigram_c1',
            'tfidf': {
                'max_features': 5000,
                'ngram_range': (1, 1),
                'min_df': 2,
                'max_df': 0.95,
                'sublinear_tf': True,
            },
            'model': {'C': 1.0, 'class_weight': None}
        },
        {
            'name': 'bigram_20k_c05',
            'tfidf': {
                'max_features': 20000,
                'ngram_range': (1, 2),
                'min_df': 2,
                'max_df': 0.95,
                'sublinear_tf': True,
            },
            'model': {'C': 0.5, 'class_weight': None}
        },
        {
            'name': 'bigram_20k_c1',
            'tfidf': {
                'max_features': 20000,
                'ngram_range': (1, 2),
                'min_df': 2,
                'max_df': 0.95,
                'sublinear_tf': True,
            },
            'model': {'C': 1.0, 'class_weight': None}
        },
        {
            'name': 'bigram_20k_c2',
            'tfidf': {
                'max_features': 20000,
                'ngram_range': (1, 2),
                'min_df': 2,
                'max_df': 0.95,
                'sublinear_tf': True,
            },
            'model': {'C': 2.0, 'class_weight': None}
        },
        {
            'name': 'bigram_20k_c1_balanced',
            'tfidf': {
                'max_features': 20000,
                'ngram_range': (1, 2),
                'min_df': 2,
                'max_df': 0.95,
                'sublinear_tf': True,
            },
            'model': {'C': 1.0, 'class_weight': 'balanced'}
        },
        {
            'name': 'bigram_50k_c1',
            'tfidf': {
                'max_features': 50000,
                'ngram_range': (1, 2),
                'min_df': 2,
                'max_df': 0.95,
                'sublinear_tf': True,
            },
            'model': {'C': 1.0, 'class_weight': None}
        }
    ],
    'FINAL_SUBMISSION_NAME': 'task3_linear_svc.csv',
}

os.makedirs(CONFIG['SUBMISSIONS_DIR'], exist_ok=True)
np.random.seed(CONFIG['RANDOM_STATE'])
print('Config loaded.')
print('Tuning experiments:', len(CONFIG['TUNING_GRID']))

Config loaded.
Tuning experiments: 6


## Section 1 - Load Data

In [2]:
# Section 1 - Load Data
train_df = pd.read_csv(CONFIG['TRAIN_PATH'], sep='\t')
test_df = pd.read_csv(CONFIG['TEST_PATH'], sep='\t')

train_texts = train_df['abstract'].astype(str)
train_labels = train_df['label_id'].astype(int)
test_texts = test_df['abstract'].astype(str)
test_ids = test_df['id'].astype(int)

print(f'Train: {len(train_df)} samples')
print(f'Test:  {len(test_df)} samples')
print(f'Classes: {train_labels.nunique()}')
train_df.head(3)

Train: 139156 samples
Test:  34790 samples
Classes: 39


,id,abstract,label_id
0,95829,Project-based learning plays a crucial role in...,14
1,73195,Edge computing is a distributed computing pa...,10
2,22319,"In today's computing environment, where Arti...",3


## Section 2 - Train/Validation Split

In [3]:
# Section 2 - Train/validation split
X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_texts,
    train_labels,
    test_size=CONFIG['VAL_SIZE'],
    random_state=CONFIG['RANDOM_STATE'],
    stratify=train_labels,
)

print(f'Train split: {len(X_train_text)} samples')
print(f'Val split:   {len(X_val_text)} samples')
print('Train class coverage:', y_train.nunique())
print('Val class coverage:  ', y_val.nunique())

Train split: 111324 samples
Val split:   27832 samples
Train class coverage: 39
Val class coverage:   39


## Section 3 - Baseline TF-IDF and LinearSVC

In [4]:
# Section 3 - Baseline model
baseline_vectorizer = TfidfVectorizer(**CONFIG['BASELINE_TFIDF'])
X_train_base = baseline_vectorizer.fit_transform(X_train_text)
X_val_base = baseline_vectorizer.transform(X_val_text)

baseline_model = LinearSVC(C=1.0, class_weight=None, random_state=CONFIG['RANDOM_STATE'])
start = time.time()
baseline_model.fit(X_train_base, y_train)
baseline_train_time = time.time() - start

baseline_val_pred = baseline_model.predict(X_val_base)
baseline_macro_f1 = f1_score(y_val, baseline_val_pred, average='macro')
baseline_acc = accuracy_score(y_val, baseline_val_pred)

print('Baseline vectorizer:', CONFIG['BASELINE_TFIDF'])
print(f'Baseline train matrix: {X_train_base.shape}')
print(f'Baseline val matrix:   {X_val_base.shape}')
print(f'Baseline train time: {baseline_train_time:.2f}s')
print(f'Baseline val Macro F1: {baseline_macro_f1:.4f}')
print(f'Baseline val Accuracy: {baseline_acc:.4f}')

Baseline vectorizer: {'max_features': 5000, 'ngram_range': (1, 1), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True}
Baseline train matrix: (111324, 5000)
Baseline val matrix:   (27832, 5000)
Baseline train time: 41.07s
Baseline val Macro F1: 0.6092
Baseline val Accuracy: 0.7186


## Section 4 - Hyperparameter Tuning

In [5]:
# Section 4 - Tuning sweep
results = []

for experiment in CONFIG['TUNING_GRID']:
    vec = TfidfVectorizer(**experiment['tfidf'])
    X_train_vec = vec.fit_transform(X_train_text)
    X_val_vec = vec.transform(X_val_text)

    model = LinearSVC(
        C=experiment['model']['C'],
        class_weight=experiment['model']['class_weight'],
        random_state=CONFIG['RANDOM_STATE']
    )

    start = time.time()
    model.fit(X_train_vec, y_train)
    train_time = time.time() - start

    y_val_pred = model.predict(X_val_vec)
    macro_f1 = f1_score(y_val, y_val_pred, average='macro')
    accuracy = accuracy_score(y_val, y_val_pred)

    row = {
        'experiment_name': experiment['name'],
        'max_features': experiment['tfidf']['max_features'],
        'ngram_range': experiment['tfidf']['ngram_range'],
        'min_df': experiment['tfidf']['min_df'],
        'max_df': experiment['tfidf']['max_df'],
        'sublinear_tf': experiment['tfidf']['sublinear_tf'],
        'C': experiment['model']['C'],
        'class_weight': experiment['model']['class_weight'],
        'val_macro_f1': macro_f1,
        'val_accuracy': accuracy,
        'train_time_sec': train_time,
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
results_df

,experiment_name,max_features,ngram_range,min_df,max_df,sublinear_tf,C,class_weight,val_macro_f1,val_accuracy,train_time_sec
0,bigram_50k_c1,50000,"(1, 2)",2,0.95,True,1.0,None,0.644714,0.742994,68.009437
1,bigram_20k_c05,20000,"(1, 2)",2,0.95,True,0.5,None,0.634766,0.741628,56.739373
2,bigram_20k_c1_balanced,20000,"(1, 2)",2,0.95,True,1.0,balanced,0.630809,0.729592,61.432175
3,bigram_20k_c1,20000,"(1, 2)",2,0.95,True,1.0,None,0.629109,0.730921,60.068319
4,bigram_20k_c2,20000,"(1, 2)",2,0.95,True,2.0,None,0.614900,0.716657,64.550517
5,baseline_unigram_c1,5000,"(1, 1)",2,0.95,True,1.0,None,0.609179,0.718561,40.093572


## Section 5 - Select Best Configuration

In [6]:
# Section 5 - Best experiment
best_result = results_df.iloc[0].to_dict()
best_experiment = next(
    exp for exp in CONFIG['TUNING_GRID']
    if exp['name'] == best_result['experiment_name']
)

print('Best experiment:')
print(best_result)

Best experiment:
{'experiment_name': 'bigram_50k_c1', 'max_features': 50000, 'ngram_range': (1, 2), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True, 'C': 1.0, 'class_weight': None, 'val_macro_f1': 0.6447137107181449, 'val_accuracy': 0.742993676343777, 'train_time_sec': 68.009437084198}


## Section 6 - Retrain Best Model on Full Training Data

In [7]:
# Section 6 - Full retraining
final_vectorizer = TfidfVectorizer(**best_experiment['tfidf'])
X_train_full = final_vectorizer.fit_transform(train_texts)
X_test_final = final_vectorizer.transform(test_texts)

final_model = LinearSVC(
    C=best_experiment['model']['C'],
    class_weight=best_experiment['model']['class_weight'],
    random_state=CONFIG['RANDOM_STATE']
)

start = time.time()
final_model.fit(X_train_full, train_labels)
final_train_time = time.time() - start
test_preds = final_model.predict(X_test_final)

print(f'Full train matrix: {X_train_full.shape}')
print(f'Test matrix:       {X_test_final.shape}')
print(f'Final train time: {final_train_time:.2f}s')
print('Test predictions generated.')

Full train matrix: (139156, 50000)
Test matrix:       (34790, 50000)
Final train time: 86.96s
Test predictions generated.


## Section 7 - Generate Submission CSV

In [8]:
# Section 7 - Save submission locally
submission = pd.DataFrame({
    'id': test_ids,
    'label_id': test_preds,
})

out_path = os.path.join(CONFIG['SUBMISSIONS_DIR'], CONFIG['FINAL_SUBMISSION_NAME'])
submission.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {submission.shape}')
submission.head()

Saved: submissions/task3_linear_svc.csv
Shape: (34790, 2)


,id,label_id
0,173148,38
1,29098,4
2,28211,4
3,136101,0
4,97133,14


## Section 8 - Results Summary

In [12]:
# Section 8 - Summary
print('=== Task 3 LinearSVC Summary ===')
print(f"Best local validation Macro F1: {best_result['val_macro_f1']:.4f}")
print(f"Best local validation Accuracy: {best_result['val_accuracy']:.4f}")
print(f"Best experiment name: {best_result['experiment_name']}")

=== Task 3 LinearSVC Summary ===
Best local validation Macro F1: 0.6447
Best local validation Accuracy: 0.7430
Best experiment name: bigram_50k_c1
